# 챕터 5 — 반도체의 세 층: 칩, 장비, 소재 (2007–2025)

KCSDB2에서 **반도체 관련 품목만 떼어내** 교역 구조를 인과 해석 없이 서술하는 기술통계 노트북이다. 반도체는 완성 **칩**(메모리 등), 그 칩을 만드는 **제조장비**, 공정에 쓰는 **소재**의 세 층으로 나뉜다. 함께 있는 문서 `chapter05.md`가 이 결과를 이야기로 엮은 것이다.

## 이 데이터베이스에 대하여 (출처·집계 기준)

이 데이터베이스는 관세청이 OpenAPI(품목별 국가별 수출입실적(GW), https://www.data.go.kr/data/15100475/openapi.do)로 공개하는 월별 수출입 통계를 **2007년 1월부터 2026년 3월까지** 한데 모아 하나의 파일로 만든 것이다.

**수치의 집계 기준(관세청 정의).** 수출입 신고 통관 자료를 국가 및 HS Code(2·4·6·10단위)별로 집계한 국가별 품목별 무역통계다. 금액은 미화(USD)이며, 수출은 FOB(신고금액), 수입은 CIF(과세가격) 기준이다. 중량은 순중량(kg). 국가는 수출은 최종목적국, 수입은 원산국을 원칙으로 하며 무역통계부호상 ISO 코드로 분류한다. 단순 통과물품이나 일시 반입·반출 물품은 제외된다(물적 자원의 증감이 없으므로). 통계는 매월 수출입 신고의 정정·취하를 반영해 전월까지 자료를 현행화한다(주기 1개월).

## 0. HS 코드 정의와 규칙

- **칩**: HS 8541(개별소자)·8542(집적회로). DRAM·HBM 등 메모리는 8542.
- **제조장비(9개 HS6)**: 848610·848620·848640·903082·903141·903149·903180·842121·848690.
- **소재(26개 HS10)**: 블랭크마스크·포토레지스트·현상제·공정가스·스퍼터링 타깃·실리콘 웨이퍼·슬러리 등.
- 장비·소재 코드는 관세청 '반도체 HS 표준해석 지침'(2023)을 정리한 **윤승환(2024), 질서경제저널 27(3)**의 분류를 따름.
- **한계**: HS는 반도체 전용이 아니다(842121 등에 비반도체 혼입, 8541에 태양전지·LED 포함). "반도체 관련"의 근사다. 2026 제외, EU 집계코드 제외.

In [ ]:
import os
import duckdb
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

EQUIP = ('848610','848620','848640','903082','903141','903149','903180','842121','848690')
MAT = ('2812190000','2812902000','2812909000','2814100000','2843301000','2923900000','3208201030',
       '3405400000','3701991000','3707901010','3707902090','3814002110','3818001000','3824997100',
       '3919900000','3926909000','6804229000','6815191000','6815990000','7006009000','7115901090',
       '7407100000','7419809000','7616999090','8103990000','8108909000')
eq = ','.join("'"+c+"'" for c in EQUIP)
mt = ','.join("'"+c+"'" for c in MAT)
CHIP = "SUBSTR(hs10,1,4) IN ('8541','8542')"
EQC  = f"SUBSTR(hs10,1,6) IN ({eq})"
MTC  = f"hs10 IN ({mt})"

DB_PATH = os.path.join("data", "processed", "kcsdb.duckdb")
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"DB 없음: {DB_PATH} — Releases에서 받아 data/processed/ 에 배치")
con = duckdb.connect(DB_PATH, read_only=True)
def q(sql): return con.sql(sql).df()
print("연결 완료 —", f"{con.sql('SELECT COUNT(*) FROM fact_trade').fetchone()[0]:,}", "거래행")

## 3. 세 층의 2025년 — 칩은 흑자, 장비·소재는 적자

In [ ]:
rows=[]
for nm,c in [('칩(8541·8542)',CHIP),('제조장비',EQC),('소재',MTC)]:
    d=q(f"SELECT ROUND(SUM(exp_dlr)/1e8,0) 수출, ROUND(SUM(imp_dlr)/1e8,0) 수입, ROUND(SUM(exp_dlr-imp_dlr)/1e8,0) 수지 FROM fact_trade WHERE yyyymm//100=2025 AND ({c})")
    rows.append((nm, d['수출'][0], d['수입'][0], d['수지'][0]))
import pandas as pd
tbl=pd.DataFrame(rows, columns=['층','수출_억','수입_억','수지_억']); print(tbl.to_string(index=False))
tot=con.sql('SELECT SUM(exp_dlr) FROM fact_trade WHERE yyyymm//100=2025').fetchone()[0]
chip=con.sql(f'SELECT SUM(exp_dlr) FROM fact_trade WHERE yyyymm//100=2025 AND ({CHIP})').fetchone()[0]
print('칩 수출 = 전체 수출의', round(100*chip/tot,1), '%')

x=np.arange(3); w=0.27
f=plt.figure(figsize=(8,4)); ax=f.gca()
ax.bar(x-w, tbl['수출_억'], w, label='Exports', color='seagreen')
ax.bar(x, tbl['수입_억'], w, label='Imports', color='indianred')
ax.bar(x+w, tbl['수지_억'], w, label='Balance', color='steelblue')
ax.axhline(0,color='black',lw=.7); ax.set_xticks(x); ax.set_xticklabels(['Chips','Equipment','Materials'])
ax.set_ylabel('2025 (100M USD)'); ax.set_title('Korea Semiconductor Trade by Layer (2025)')
ax.legend(fontsize=8); ax.grid(alpha=.3,axis='y'); f.tight_layout(); plt.show()

## 4. 20년의 궤적 — 칩은 치솟고, 장비 적자는 그대로

In [ ]:
ts=q(f'''SELECT yyyymm//100 yr,
 ROUND(SUM(CASE WHEN {CHIP} THEN exp_dlr-imp_dlr END)/1e8,0) 칩,
 ROUND(SUM(CASE WHEN {EQC} THEN exp_dlr-imp_dlr END)/1e8,0) 장비,
 ROUND(SUM(CASE WHEN {MTC} THEN exp_dlr-imp_dlr END)/1e8,0) 소재
 FROM fact_trade WHERE yyyymm//100<2026 GROUP BY 1 ORDER BY 1''')
share=q(f'''SELECT yyyymm//100 yr, ROUND(100.0*SUM(CASE WHEN {CHIP} THEN exp_dlr END)/SUM(exp_dlr),1) pct
 FROM fact_trade WHERE yyyymm//100<2026 GROUP BY 1 ORDER BY 1''')
print('칩 수출 비중: 2007', share['pct'].iloc[0], '% -> 2025', share['pct'].iloc[-1], '%')

f=plt.figure(figsize=(8,3.8)); ax=f.gca()
ax.plot(ts['yr'], ts['칩'], marker='o', ms=3, label='Chips')
ax.plot(ts['yr'], ts['장비'], marker='s', ms=3, label='Equipment')
ax.plot(ts['yr'], ts['소재'], marker='^', ms=3, label='Materials')
ax.axhline(0,color='black',lw=.7)
ax.set_ylabel('Trade balance (100M USD)'); ax.set_xlabel('Year')
ax.set_title('Semiconductor Trade Balance by Layer (2007-2025)')
ax.legend(fontsize=8); ax.grid(alpha=.3); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
f.tight_layout(); plt.show()

## 5. 어디서 사고 어디에 파나

In [ ]:
EN={'미합중국':'USA','중국':'China','일본':'Japan','네덜란드':'Netherlands','대만':'Taiwan',
    '홍콩':'HK','베트남':'Vietnam','싱가포르':'Singapore','독일':'Germany','말레이시아':'Malaysia',
    '필리핀':'Philippines','이스라엘':'Israel'}
def L(k): return EN.get(k,k)
def partners(cond, col, n=8):
    return q(f'''SELECT COALESCE(c.name_ko_mofa,c.name_ko_kcs) 국가, ROUND(SUM(f.{col})/1e8,0) 억
        FROM fact_trade f JOIN dim_country c USING(stat_cd)
        WHERE f.yyyymm//100=2025 AND ({cond}) AND c.stat_cd<>'EU'
        GROUP BY 1 ORDER BY 억 DESC LIMIT {n}''')
print('■ 칩 수출 상위'); print(partners(CHIP,'exp_dlr').to_string(index=False))
print('\n■ 장비 수입 상위'); print(partners(EQC,'imp_dlr').to_string(index=False))
print('\n■ 소재 수입 상위'); print(partners(MTC,'imp_dlr').to_string(index=False))

# 칩 수출 상대국
dc=partners(CHIP,'exp_dlr',7)
f=plt.figure(figsize=(7,3.4)); ax=f.gca()
ax.barh([L(x) for x in dc['국가']][::-1], list(dc['억'])[::-1], color='seagreen')
ax.set_xlabel('Chip exports 2025 (100M USD)')
ax.set_title('Where Korea exports chips (2025)')
ax.grid(alpha=.3,axis='x'); f.tight_layout(); plt.show()

# 장비 수입 상대국
d=partners(EQC,'imp_dlr',7)
f=plt.figure(figsize=(7,3.4)); ax=f.gca()
ax.barh([L(x) for x in d['국가']][::-1], list(d['억'])[::-1], color='indianred')
ax.set_xlabel('Equipment imports 2025 (100M USD)')
ax.set_title('Where Korea imports chip-making equipment (2025)')
ax.grid(alpha=.3,axis='x'); f.tight_layout(); plt.show()

## 마무리

한국 반도체는 **칩은 세계 최강 흑자(2025년 +799억, 전체 수출의 20.7%)지만, 그 칩을 만드는 장비(−147억)와 소재(−7억)는 수입 의존 적자**다. 칩은 아시아(중국·대만·베트남·홍콩)에 팔고, 장비는 네덜란드(ASML)·일본·미국에서, 소재는 일본 등에서 사온다. 모든 셀은 관측된 구조를 서술할 뿐 인과를 주장하지 않는다.

한계: HS 범위는 관세청 지침 근사(비반도체 혼입 가능, 8541에 태양전지·LED 포함), 최종목적국 왜곡(홍콩 재수출), EU 제외, 2026 제외. HS 분류 출처: 윤승환(2024), 질서경제저널 27(3). 상세는 `chapter05.md` 참조.

In [ ]:
con.close()
print("연결 종료.")